In [95]:
%%writefile solucao.py
import heapq
import sys
import time

AGENTE, VAZIO, PAREDE, ALVO = '🙎', '⚪️', '🧱', '🟢'

class SokobanIA:
    def __init__(self, filename):
        with open(filename, 'r', encoding='utf-8') as f:
            content = f.read().split()
        self.dim = int(len(content)**0.5)
        self.grid_estatico, self.caixas_init, self.alvos = [], {}, []
        self.pos_agente_init = None

        idx = 0
        for r in range(self.dim):
            linha = []
            for c in range(self.dim):
                char = content[idx]
                clean = char.replace('️⃣', '')
                if char == PAREDE: linha.append(PAREDE)
                elif char == ALVO:
                    linha.append(ALVO); self.alvos.append((r, c))
                else:
                    linha.append(VAZIO)
                    if char == AGENTE: self.pos_agente_init = (r, c)
                    elif clean.isdigit(): self.caixas_init[(r, c)] = int(clean)
                idx += 1
            self.grid_estatico.append(linha)

    def h(self, caixas):
        return sum(min(abs(p[0]-t[0]) + abs(p[1]-t[1]) for t in self.alvos) for p in caixas)

    def resolver(self, modo):
        caixas_t = tuple(sorted(self.caixas_init.items()))
        h_init = self.h(self.caixas_init)
        queue = [(h_init if modo != "dijkstra" else 0, 0, self.pos_agente_init, caixas_t, "")]
        visitados = {}

        while queue:
            f, g, p_agente, caixas, caminho = heapq.heappop(queue)
            if (p_agente, caixas) in visitados and visitados[(p_agente, caixas)] <= g: continue
            visitados[(p_agente, caixas)] = g

            dict_c = dict(caixas)
            if all(pos in self.alvos for pos in dict_c): return caminho, g, dict_c, p_agente

            for dr, dc, seta in [(-1,0,'⬆️'), (1,0,'⬇️'), (0,-1,'⬅️'), (0,1,'➡️')]:
                nr, nc = p_agente[0]+dr, p_agente[1]+dc
                if 0 <= nr < self.dim and 0 <= nc < self.dim and self.grid_estatico[nr][nc] != PAREDE:
                    if p_agente in dict_c:
                        peso = dict_c[p_agente]
                        novas = dict_c.copy()
                        del novas[p_agente]
                        novas[(nr, nc)] = peso
                        self.push(queue, modo, g + 1 + peso, nr, nc, tuple(sorted(novas.items())), caminho + seta)
                    else:
                        self.push(queue, modo, g + 1, nr, nc, caixas, caminho + seta)
        return None

    def push(self, q, modo, g, nr, nc, caixas, cam):
        heur = self.h(dict(caixas))
        f = g if modo == "dijkstra" else (heur if modo == "ganancioso" else g + heur)
        heapq.heappush(q, (f, g, (nr, nc), caixas, cam))

    def salvar(self, alg, res):
        if not res: return
        cam, custo, caixas_f, pos_f = res
        with open(f"{alg}.txt", 'w', encoding='utf-8') as f:
            f.write("Estado final\n")
            for r in range(self.dim):
                linha = [str(caixas_f[(r,c)]) if (r,c) in caixas_f else (AGENTE if (r,c)==pos_f else self.grid_estatico[r][c]) for c in range(self.dim)]
                f.write(" ".join(linha) + "\n")
            f.write(f"\nMovimentos\n{cam}\n\nQuantidades de movimentos\n{len(cam)}\n")

def salvar_grid_visual(n, nome):
    matriz = [[VAZIO for _ in range(n)] for _ in range(n)]
    for i in range(n):
        matriz[0][i] = PAREDE
        matriz[n-1][i] = PAREDE
        matriz[i][0] = PAREDE
        matriz[i][n-1] = PAREDE
    matriz[1][1] = AGENTE
    matriz[n//2][n//2] = '5'
    matriz[n-2][n-2] = ALVO
    with open(nome, "w", encoding="utf-8") as f:
        for linha in matriz:
            f.write(" ".join(linha) + "\n")

if __name__ == "__main__":
    for tam in [8, 16, 24, 64]:
        salvar_grid_visual(tam, f"grid{tam}.txt")

    args_txt = [a for a in sys.argv if a.endswith('.txt')]
    args_alg = [a for a in sys.argv if a in ["dijkstra", "ganancioso", "a_estrela"]]

    arq_entrada = args_txt[0] if args_txt else "grid8.txt"
    algoritmo_escolhido = args_alg[0] if args_alg else "todos"

    print(f"\n--- Lendo Arquivo: {arq_entrada} ---")

    try:
        solver = SokobanIA(arq_entrada)
        if algoritmo_escolhido == "todos":
            algs_para_rodar = ["dijkstra", "ganancioso", "a_estrela"]
        else:
            algs_para_rodar = [algoritmo_escolhido]

        for alg in algs_para_rodar:
            print(f"Executando {alg.upper()}...")
            start = time.time()
            res = solver.resolver(alg)
            if res:
                solver.salvar(alg, res)
                print(f"Sucesso! Custo: {res[1]} | Tempo: {time.time()-start:.4f}s | Salvo em: {alg}.txt")
            else:
                print(f"Falha: {alg.upper()} não encontrou solução.")
    except FileNotFoundError:
        print(f"Erro: O arquivo '{arq_entrada}' não foi encontrado no diretório.")

Overwriting solucao.py


In [96]:
!python solucao.py grid16.txt


--- Lendo Arquivo: grid16.txt ---
Executando DIJKSTRA...
Sucesso! Custo: 86 | Tempo: 0.0065s | Salvo em: dijkstra.txt
Executando GANANCIOSO...
Sucesso! Custo: 86 | Tempo: 0.0015s | Salvo em: ganancioso.txt
Executando A_ESTRELA...
Sucesso! Custo: 86 | Tempo: 0.0045s | Salvo em: a_estrela.txt


```mermaid
graph TD
    Start((Início)) --> GetState[Ler Estado Atual]
    GetState --> CheckGoal{Objetivo atingido?}
    
    CheckGoal -- Sim --> End((Fim: Sucesso))
    
    CheckGoal -- Não --> Expand[Gerar Movimentos: N, S, L, O]
    Expand --> WallCheck{É Parede?}
    
    WallCheck -- Sim --> Discard[Descartar Movimento]
    WallCheck -- Não --> BoxCheck{Agente está na Caixa?}
    
    BoxCheck -- Sim --> CostWeighted[Custo = 1 + Peso]
    BoxCheck -- Não --> CostNormal[Custo = 1]
    
    CostWeighted --> CalcHeuristic[Calcular f n = g n + h n]
    CostNormal --> CalcHeuristic
    
    CalcHeuristic --> PriorityQueue[Inserir na Fila de Prioridade]
    PriorityQueue --> NextState[Extrair estado com menor f n]
    NextState --> GetState
```